In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('../data/deliverylens.db')

orders = pd.read_sql("SELECT * FROM orders", conn)
items = pd.read_sql("SELECT * FROM items", conn)
reviews = pd.read_sql("SELECT * FROM reviews", conn)

In [2]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print("Dates fixed!")
print(orders.dtypes)

C:\Users\addys\AppData\Local\Temp\ipykernel_24248\2907371498.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders[col] = pd.to_datetime(orders[col])
C:\Users\addys\AppData\Local\Temp\ipykernel_24248\2907371498.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders[col] = pd.to_datetime(orders[col])
C:\Users\addys\AppData\Local\Temp\ipykernel_24248\2907371498.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders[col] = pd.to_datetime(orders[col])
C:\Users\addys\AppData\Local\Temp\ipykernel_24248\2907371498.py:10: UserWarning: Could not infer format, so each

Dates fixed!
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


C:\Users\addys\AppData\Local\Temp\ipykernel_24248\2907371498.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders[col] = pd.to_datetime(orders[col])


In [3]:
# Review score 1 ya 2 = unhappy customer = return proxy
reviews['is_bad_review'] = reviews['review_score'].apply(
    lambda x: 1 if x <= 2 else 0 
)

print("Bad reviews count:", reviews['is_bad_review'].sum())
print("Total reviews:", len(reviews))

Bad reviews count: 14575
Total reviews: 99224


In [4]:
# Yeh columns kaam nahi aayenge
reviews = reviews.drop(columns=[
    'review_comment_title', 
    'review_comment_message',
    'review_answer_timestamp'
])

print("Reviews columns after drop:", reviews.columns.tolist())

Reviews columns after drop: ['review_id', 'order_id', 'review_score', 'review_creation_date', 'is_bad_review']


In [5]:
orders_items = pd.merge(orders, items, on='order_id', how='left')
print("Orders + Items shape:", orders_items.shape) 

Orders + Items shape: (113425, 14)


In [6]:
master_df = pd.merge(orders_items, reviews, on='order_id', how='left')
print("Master DataFrame shape:", master_df.shape)
master_df.head()

Master DataFrame shape: (114092, 18)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,review_id,review_score,review_creation_date,is_bad_review
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-02-10 10:56:00,2017-02-10 11:07:00,2017-04-10 19:55:00,2017-10-10 21:25:00,2017-10-18,1.0,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,a54f0611adc9ed256b57ede6b6eb5114,4.0,11-10-17 00:00,0.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:00,2018-07-26 03:24:00,2018-07-26 14:31:00,2018-07-08 15:27:00,2018-08-13,1.0,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,8d5266042046a06655c8db133d120ba5,4.0,08-08-18 00:00,0.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:00,2018-08-08 08:55:00,2018-08-08 13:50:00,2018-08-17 18:06:00,2018-04-09,1.0,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,e73b67b67587f7644d5bd1a52deb1b01,5.0,18-08-18 00:00,0.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:00,2017-11-18 19:45:00,2017-11-22 13:39:00,2017-02-12 00:28:00,2017-12-15,1.0,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20,359d03e676b3c069f62cadba8dd3f6e8,5.0,03-12-17 00:00,0.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:00,2018-02-13 22:20:00,2018-02-14 19:46:00,2018-02-16 18:17:00,2018-02-26,1.0,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72,e50934924e227544ba8246aeb3770dd4,5.0,17-02-18 00:00,0.0


In [7]:
# Kitne din late deliver hua estimated se
master_df['delivery_delay_days'] = (
    master_df['order_delivered_customer_date'] - 
    master_df['order_estimated_delivery_date']
).dt.days

print("Delivery delay column created!")
master_df['delivery_delay_days'].describe()

Delivery delay column created!


count    110839.000000
mean        -11.377845
std         115.838599
min        -537.000000
25%         -65.000000
50%         -11.000000
75%          43.000000
max         347.000000
Name: delivery_delay_days, dtype: float64

In [8]:
master_df.to_csv('../data/deliverylens_clean.csv', index=False)
print("Clean data saved!")

Clean data saved!
